In [11]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models

In [12]:
SEQUENCE_LENGTH = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "resnet50_lstm_best.pth"
VIDEOS_DIR = "videos"

classes = ['abuse', 'arrest', 'arson', 'assault', 'burglary', 'explosion',
           'fighting', 'normal', 'roadaccidents', 'robbery', 'shooting',
           'shoplifting', 'stealing', 'vandalism']

print("Using device:", DEVICE)

Using device: cpu


In [13]:
class ResNet50LSTM(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super().__init__()

        backbone = models.resnet50(weights=None)
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])

        feature_dim = 2048

        self.lstm = nn.LSTM(
            input_size=feature_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)

        features = self.feature_extractor(x)
        features = features.view(B, T, -1)

        _, (h_n, _) = self.lstm(features)
        final_hidden = h_n[-1]

        return self.fc(final_hidden)

model = ResNet50LSTM(num_classes=len(classes)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("Model loaded successfully!")

Model loaded successfully!


In [14]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def sample_frames(video_path, sequence_length=16):
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        raise RuntimeError(f"Invalid video: {video_path}")

    indices = np.linspace(0, frame_count - 1, sequence_length).astype(int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            if len(frames) > 0:
                frame = frames[-1]
            else:
                frame = np.zeros((224, 224, 3), dtype=np.uint8)

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return frames

def predict_video(video_path):
    frames = sample_frames(video_path, SEQUENCE_LENGTH)
    frames = [transform(f) for f in frames]
    frames = torch.stack(frames).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output = model(frames)
        probabilities = torch.softmax(output, dim=1)[0]
        pred_idx = output.argmax(1).item()
        confidence = probabilities[pred_idx].item() * 100

    return classes[pred_idx], confidence

def get_actual_label(filename):
    """Extract the actual label from the filename (e.g., 'Abuse001_x264.mp4' -> 'abuse')"""
    name = filename.split('_')[0]
    label = ''.join([c for c in name if not c.isdigit()]).lower()
    return label

In [15]:
# Classify all videos in the 'videos' folder
results = []

video_files = sorted([
    f for f in os.listdir(VIDEOS_DIR)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
])

print(f"Found {len(video_files)} videos in '{VIDEOS_DIR}/'\n")
print(f"{'Video File':40s} {'Actual':18s} {'Predicted':18s} {'Confidence':>10s}  Result")
print(f"{'-'*100}")

for video_file in video_files:
    video_path = os.path.join(VIDEOS_DIR, video_file)
    actual_label = get_actual_label(video_file)

    try:
        prediction, confidence = predict_video(video_path)
        is_correct = prediction == actual_label
        status = "CORRECT" if is_correct else "WRONG"

        print(f"{video_file:40s} {actual_label:18s} {prediction:18s} {confidence:9.1f}%  {status}")

        results.append({
            "video": video_file,
            "folder": actual_label,
            "prediction": prediction,
            "confidence": confidence,
            "correct": is_correct
        })

    except Exception as e:
        print(f"{video_file:40s} {actual_label:18s} {'ERROR':18s} {str(e)}")

Found 39 videos in 'videos/'

Video File                               Actual             Predicted          Confidence  Result
----------------------------------------------------------------------------------------------------
Abuse001_x264.mp4                        abuse              arson                   13.0%  WRONG
Abuse006_x264.mp4                        abuse              arson                   15.3%  WRONG
Abuse011_x264.mp4                        abuse              normal                  23.7%  WRONG
Arrest024_x264.mp4                       arrest             normal                  14.7%  WRONG
Arrest029_x264.mp4                       arrest             explosion               11.1%  WRONG
Arson005_x264.mp4                        arson              arson                   14.6%  CORRECT
Arson027_x264.mp4                        arson              arson                   14.9%  CORRECT
Assault015_x264.mp4                      assault            roadaccidents           10.8

In [16]:
# Summary
total = len(results)
correct = sum(1 for r in results if r["correct"])

print(f"\n{'='*60}")
print(f"  SUMMARY")
print(f"{'='*60}")
print(f"  Total Videos  : {total}")
print(f"  Correct       : {correct}")
print(f"  Wrong         : {total - correct}")
print(f"  Accuracy      : {correct/total*100:.1f}%" if total > 0 else "  No videos found.")


  SUMMARY
  Total Videos  : 39
  Correct       : 7
  Wrong         : 32
  Accuracy      : 17.9%
